# 🌫️ Indian Urban Air Quality Intelligence System
### ML-Powered AQI Prediction & Health Risk Analysis — 24 Cities | 2019–2024

---

**Project Summary**

This end-to-end data science project applies the **India CPCB sub-index methodology** to compute scientifically grounded AQI values from raw pollutant concentrations, then trains multiple machine learning models to:
1. **Predict AQI** values from pollutant readings (Random Forest & Gradient Boosting)
2. **Classify AQI categories** (Good → Severe) with 99.7% accuracy
3. **Score health risk** (0–100) for every city observation

All outputs are exported to Power BI-ready CSVs for interactive dashboard creation.

| Metric | Value |
|--------|-------|
| Dataset | 10,000 records, 24 Indian cities |
| Features | 18 (13 raw + 5 engineered) |
| AQI Regression R² | **0.9999** |
| Category Classifier Accuracy | **99.7%** |
| Health Risk Model R² | **0.9991** |

In [ ]:
# ── Imports ──────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import accuracy_score, r2_score, mean_absolute_error, confusion_matrix
from sklearn.pipeline import Pipeline

print('All libraries loaded ✓')

## 1. Data Loading & Exploration

In [ ]:
df = pd.read_csv('data/raw_data.csv')
print(f'Shape: {df.shape}')
print(f'Cities: {df["City"].nunique()} | Nulls: {df.isnull().sum().sum()}')
df.head()

In [ ]:
df.describe().round(2)

## 2. Feature Engineering

### 2.1 CPCB AQI Sub-Index Methodology

India's Central Pollution Control Board (CPCB) computes AQI as the **maximum sub-index** across pollutants, using breakpoint tables for PM2.5, PM10, NO2, SO2, CO, and O3. This ensures the most hazardous pollutant drives the overall AQI rating.

In [ ]:
def si_pm25(c):
    """PM2.5 sub-index — CPCB breakpoints (µg/m³)"""
    bp = [(0,30,0,50),(30,60,51,100),(60,90,101,200),
          (90,120,201,300),(120,250,301,400),(250,380,401,500)]
    for lo,hi,ilo,ihi in bp:
        if lo <= c <= hi: return ilo + (c-lo)/(hi-lo)*(ihi-ilo)
    return 500

def si_pm10(c):
    """PM10 sub-index — CPCB breakpoints (µg/m³)"""
    bp = [(0,50,0,50),(50,100,51,100),(100,250,101,200),
          (250,350,201,300),(350,430,301,400),(430,600,401,500)]
    for lo,hi,ilo,ihi in bp:
        if lo <= c <= hi: return ilo + (c-lo)/(hi-lo)*(ihi-ilo)
    return 500

def si_no2(c):
    """NO2 sub-index — CPCB breakpoints (µg/m³)"""
    bp = [(0,40,0,50),(40,80,51,100),(80,180,101,200),
          (180,280,201,300),(280,400,301,400),(400,800,401,500)]
    for lo,hi,ilo,ihi in bp:
        if lo <= c <= hi: return ilo + (c-lo)/(hi-lo)*(ihi-ilo)
    return 500

df['SI_PM25'] = df['PM2.5'].apply(si_pm25)
df['SI_PM10'] = df['PM10'].apply(si_pm10)
df['SI_NO2']  = df['NO2'].apply(si_no2)

# AQI = max sub-index (CPCB rule)
df['AQI_Computed'] = df[['SI_PM25','SI_PM10','SI_NO2']].max(axis=1).clip(0,500).round(0).astype(int)

def aqi_cat(v):
    if v <= 50: return 'Good'
    elif v <= 100: return 'Satisfactory'
    elif v <= 200: return 'Moderate'
    elif v <= 300: return 'Poor'
    elif v <= 400: return 'Very Poor'
    else: return 'Severe'

df['AQI_Category'] = df['AQI_Computed'].apply(aqi_cat)
print('CPCB AQI computed ✓')
df[['PM2.5','PM10','NO2','SI_PM25','SI_PM10','SI_NO2','AQI_Computed','AQI_Category']].head(10)

In [ ]:
# Additional engineered features
df['Pollution_Load']  = (df['PM2.5']*0.40 + df['PM10']*0.25 + df['NO2']*0.15 +
                          df['SO2']*0.10 + df['CO']*0.07 + df['O3']*0.03)
df['Climate_Stress']  = (df['Temperature (°C)']*0.4 + df['Humidity (%)']*0.3 -
                          df['Wind Speed (km/h)']*0.2 - df['Rainfall (mm)']*0.1)
df['Urban_Pressure']  = (df['Vehicle Count']/1e5) * df['Industrial Activity Index']
df['PM_Ratio']        = df['PM2.5'] / (df['PM10'] + 1)
df['Oxidant_Load']    = df['O3'] + df['NO2']

metros = ['Delhi','Mumbai','Kolkata','Bangalore','Chennai','Hyderabad']
df['City_Tier'] = df['City'].apply(lambda c: 'Metro' if c in metros else 'Tier-2')

print('Engineered features:', ['Pollution_Load','Climate_Stress','Urban_Pressure','PM_Ratio','Oxidant_Load'])

## 3. Exploratory Data Analysis

In [ ]:
PALETTE = {'Good':'#2ECC71','Satisfactory':'#F1C40F','Moderate':'#E67E22',
           'Poor':'#E74C3C','Very Poor':'#8E44AD','Severe':'#2C3E50'}
cat_order = ['Good','Satisfactory','Moderate','Poor','Very Poor','Severe']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
counts = df['AQI_Category'].value_counts().reindex(cat_order, fill_value=0)
axes[0].bar(cat_order, counts.values, color=[PALETTE[c] for c in cat_order], edgecolor='white')
axes[0].set_title('AQI Category Distribution (CPCB)', fontweight='bold')
axes[0].set_xlabel('Category'); axes[0].set_ylabel('Count')

axes[1].hist(df['AQI_Computed'], bins=50, color='#3498DB', edgecolor='white', alpha=0.85)
axes[1].axvline(df['AQI_Computed'].mean(), color='red', linestyle='--', lw=2,
                label=f'Mean={df["AQI_Computed"].mean():.0f}')
axes[1].set_title('AQI Distribution', fontweight='bold')
axes[1].legend()
plt.suptitle('AQI Overview — 24 Indian Cities', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# City-level AQI heatmap
city_cat = df.groupby(['City','AQI_Category']).size().unstack(fill_value=0)
city_cat = city_cat.reindex(columns=cat_order, fill_value=0)
city_cat_pct = city_cat.div(city_cat.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(14, 9))
sns.heatmap(city_cat_pct, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax,
            linewidths=0.5, cbar_kws={'label':'% of Records'})
ax.set_title('AQI Category Distribution by City (%)', fontsize=14, fontweight='bold')
ax.set_xlabel('AQI Category'); ax.set_ylabel('City')
plt.tight_layout(); plt.show()

In [ ]:
# Correlation matrix
pollutants = ['AQI_Computed','PM2.5','PM10','NO2','CO','SO2','O3',
              'Pollution_Load','Urban_Pressure','Oxidant_Load']
fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones(len(pollutants), dtype=bool))
sns.heatmap(df[pollutants].corr(), mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, ax=ax)
ax.set_title('Correlation Matrix — Pollutants & Engineered Features', fontweight='bold')
plt.tight_layout(); plt.show()

## 4. Machine Learning Models

### 4.1 AQI Regression — Predicting exact AQI value

In [ ]:
FEATURES = ['PM2.5','PM10','NO2','CO','SO2','O3',
            'Temperature (°C)','Humidity (%)','Wind Speed (km/h)',
            'Rainfall (mm)','Pressure (hPa)','Vehicle Count',
            'Industrial Activity Index','Pollution_Load',
            'Climate_Stress','Urban_Pressure','PM_Ratio','Oxidant_Load']

X = df[FEATURES]
y = df['AQI_Computed']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {len(X_train):,} | Test: {len(X_test):,}')

In [ ]:
from sklearn.metrics import mean_squared_error

rf_reg = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  RandomForestRegressor(n_estimators=200, max_depth=20,
                                      min_samples_split=5, random_state=42, n_jobs=-1))
])
rf_reg.fit(X_train, y_train)
y_pred = rf_reg.predict(X_test)

r2   = r2_score(y_test, y_pred)
mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f'Random Forest Regressor')
print(f'  R²   = {r2:.4f}')
print(f'  MAE  = {mae:.2f}')
print(f'  RMSE = {rmse:.2f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_test, y_pred, alpha=0.3, s=8, color='#3498DB')
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
axes[0].plot(lims, lims, 'r--', lw=2, label='Perfect Prediction')
axes[0].set_title(f'Actual vs Predicted AQI  (R²={r2:.4f})', fontweight='bold')
axes[0].set_xlabel('Actual AQI'); axes[0].set_ylabel('Predicted AQI'); axes[0].legend()

fi = rf_reg.named_steps['model'].feature_importances_
fi_df = pd.DataFrame({'Feature':FEATURES,'Importance':fi}).sort_values('Importance').tail(12)
axes[1].barh(fi_df['Feature'], fi_df['Importance'], color='#3498DB', alpha=0.85)
axes[1].set_title('Top Feature Importances', fontweight='bold')
axes[1].set_xlabel('Importance Score')

plt.tight_layout(); plt.show()

### 4.2 AQI Category Classifier (Good → Severe)

In [ ]:
le = LabelEncoder()
y_cls = le.fit_transform(df['AQI_Category'])

X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(
    X, y_cls, test_size=0.2, random_state=42, stratify=y_cls)

rf_cls = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  RandomForestClassifier(n_estimators=200, max_depth=20,
                                       random_state=42, n_jobs=-1))
])
rf_cls.fit(X_tr_c, y_tr_c)
y_pred_c = rf_cls.predict(X_te_c)

acc = accuracy_score(y_te_c, y_pred_c)
cv  = cross_val_score(rf_cls, X, y_cls, cv=5, scoring='accuracy', n_jobs=-1)

print(f'Accuracy      : {acc*100:.2f}%')
print(f'CV Accuracy   : {cv.mean()*100:.2f}% ± {cv.std()*100:.2f}%')
print(f'\nClassification Report:')
from sklearn.metrics import classification_report
print(classification_report(y_te_c, y_pred_c, target_names=le.classes_))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
cm = confusion_matrix(y_te_c, y_pred_c)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=le.classes_, yticklabels=le.classes_)
ax.set_title(f'Confusion Matrix — {acc*100:.1f}% Accuracy', fontweight='bold')
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.tight_layout(); plt.show()

### 4.3 Health Risk Score Model (0–100 scale)

In [ ]:
df['Health_Risk_Score'] = (
    (df['AQI_Computed']/500)*50 +
    (df['PM2.5']/250)*25 +
    (df['Pollution_Load']/df['Pollution_Load'].max())*15 +
    (df['Urban_Pressure']/df['Urban_Pressure'].max())*10
).clip(0,100)

y_hr = df['Health_Risk_Score']
X_tr_hr, X_te_hr, y_tr_hr, y_te_hr = train_test_split(X, y_hr, test_size=0.2, random_state=42)

rf_hr = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  RandomForestRegressor(n_estimators=200, max_depth=20, random_state=42, n_jobs=-1))
])
rf_hr.fit(X_tr_hr, y_tr_hr)
y_pred_hr = rf_hr.predict(X_te_hr)

r2_hr  = r2_score(y_te_hr, y_pred_hr)
mae_hr = mean_absolute_error(y_te_hr, y_pred_hr)
print(f'Health Risk Model → R²={r2_hr:.4f} | MAE={mae_hr:.2f}')

In [ ]:
city_risk = df.groupby('City')['Health_Risk_Score'].mean().sort_values(ascending=False)
colors = ['#E74C3C' if v>=50 else '#E67E22' if v>=35 else '#F1C40F' for v in city_risk]

fig, ax = plt.subplots(figsize=(14, 6))
ax.bar(city_risk.index, city_risk.values, color=colors, edgecolor='white')
ax.axhline(50, color='red', linestyle='--', lw=2, label='High Risk Threshold')
ax.set_title('Health Risk Score by City', fontsize=14, fontweight='bold')
ax.set_xlabel('City'); ax.set_ylabel('Health Risk Score (0–100)')
plt.xticks(rotation=45, ha='right'); ax.legend()
plt.tight_layout(); plt.show()

## 5. Export for Power BI

In [ ]:
import os
os.makedirs('powerbi_data', exist_ok=True)

df['AQI_Predicted']         = rf_reg.predict(X)
df['AQI_Cat_Predicted']     = le.inverse_transform(rf_cls.predict(X))
df['Health_Risk_Predicted'] = rf_hr.predict(X)

df.to_csv('powerbi_data/fact_aqi_enriched.csv', index=False)

city_summary = df.groupby('City').agg(
    Avg_AQI=('AQI_Computed','mean'), Max_AQI=('AQI_Computed','max'),
    Avg_PM25=('PM2.5','mean'), Avg_Health_Risk=('Health_Risk_Score','mean'),
    Records=('AQI_Computed','count'),
    City_Tier=('City_Tier','first'),
).reset_index().round(2)
city_summary.to_csv('powerbi_data/dim_city_summary.csv', index=False)

print('Power BI CSVs exported ✓')
print('Files: fact_aqi_enriched.csv, dim_city_summary.csv')

## 6. Key Insights

| Insight | Finding |
|---------|----------|
| **Dominant Pollutant** | PM10 is the primary AQI driver for most Indian cities |
| **Most Polluted** | Ahmedabad, Kolkata, Thane show consistently highest AQI |
| **Cleanest Cities** | Ludhiana, Srinagar, Bhopal have relatively lower AQI |
| **Metro vs Tier-2** | No significant AQI gap — industrial activity affects Tier-2 cities too |
| **Health Risk** | Cities with high PM2.5 + industrial activity face compounded health risk |
| **Model Accuracy** | CPCB-grounded targets allow near-perfect ML performance (R²=0.9999) |

---
*Built with Python, scikit-learn, and visualised in Power BI*